In [1]:
import os
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

BASE = r"C:\Users\padhi\Downloads\siH26052_data"
FEATURE_DIR = os.path.join(BASE, "features")
MODEL_DIR = os.path.join(BASE, "model")

os.makedirs(MODEL_DIR, exist_ok=True)

print("Ready")

Ready


In [2]:
X_train_full = np.load(
    os.path.join(FEATURE_DIR, "X_train.npy"),
    mmap_mode="r"
)

y_train_full = np.load(
    os.path.join(FEATURE_DIR, "y_train.npy"),
    mmap_mode="r"
)

X_dev = np.load(
    os.path.join(FEATURE_DIR, "X_dev.npy"),
    mmap_mode="r"
)

y_dev = np.load(
    os.path.join(FEATURE_DIR, "y_dev.npy"),
    mmap_mode="r"
)

print("X_train:", X_train_full.shape)
print("y_train:", y_train_full.shape)

print("X_dev:", X_dev.shape)
print("y_dev:", y_dev.shape)

X_train: (3217511, 257)
y_train: (3217511, 257)
X_dev: (241529, 257)
y_dev: (241529, 257)


In [3]:
N_SAMPLES = 500_000

rng = np.random.default_rng(42)

indices = rng.choice(
    X_train_full.shape[0],
    size=N_SAMPLES,
    replace=False
)

X_train = X_train_full[indices]
y_train = y_train_full[indices]

print("Selected X:", X_train.shape)
print("Selected y:", y_train.shape)

Selected X: (500000, 257)
Selected y: (500000, 257)


In [4]:
scaler_X = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)

X_dev_scaled = scaler_X.transform(X_dev)

print("X scaling complete")

X scaling complete


In [5]:
scaler_y = StandardScaler()

y_train_scaled = scaler_y.fit_transform(y_train)

print("y scaling complete")

y scaling complete


In [6]:
model = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    batch_size=256,
    max_iter=30,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42,
    verbose=True
)

print(model)

MLPRegressor(batch_size=256, early_stopping=True, hidden_layer_sizes=(128, 64),
             max_iter=30, random_state=42, verbose=True)


In [7]:
model.fit(
    X_train_scaled,
    y_train_scaled
)

Iteration 1, loss = 0.14013644
Validation score: 0.797879
Iteration 2, loss = 0.09500292
Validation score: 0.816438
Iteration 3, loss = 0.08837551
Validation score: 0.827154
Iteration 4, loss = 0.08642166
Validation score: 0.815793
Iteration 5, loss = 0.08547133
Validation score: 0.827187
Iteration 6, loss = 0.08479840
Validation score: 0.827122
Iteration 7, loss = 0.08432102
Validation score: 0.831586
Iteration 8, loss = 0.08412558
Validation score: 0.827611
Iteration 9, loss = 0.08380985
Validation score: 0.832731
Iteration 10, loss = 0.08374366
Validation score: 0.832790
Iteration 11, loss = 0.08345457
Validation score: 0.831570
Iteration 12, loss = 0.08339411
Validation score: 0.833865
Iteration 13, loss = 0.08321858
Validation score: 0.833608
Iteration 14, loss = 0.08315054
Validation score: 0.833019
Iteration 15, loss = 0.08289557
Validation score: 0.834855
Iteration 16, loss = 0.08294939
Validation score: 0.832747
Iteration 17, loss = 0.08273895
Validation score: 0.833495
Iterat

C:\Users\padhi\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPRegressor(batch_size=256, early_stopping=True, hidden_layer_sizes=(128, 64),
             max_iter=30, random_state=42, verbose=True)

In [8]:
y_dev_pred_scaled = model.predict(X_dev_scaled)

y_dev_pred = scaler_y.inverse_transform(
    y_dev_pred_scaled
)

print("Prediction complete")
print("Prediction shape:", y_dev_pred.shape)

Prediction complete
Prediction shape: (241529, 257)


In [9]:
mse = mean_squared_error(
    y_dev,
    y_dev_pred
)

mae = mean_absolute_error(
    y_dev,
    y_dev_pred
)


print("VALIDATION RESULTS")


print("MSE:", mse)
print("MAE:", mae)

VALIDATION RESULTS
MSE: 0.07805915176868439
MAE: 0.08119124174118042


In [10]:
joblib.dump(
    model,
    os.path.join(MODEL_DIR, "mlp_model.pkl")
)

joblib.dump(
    scaler_X,
    os.path.join(MODEL_DIR, "scaler_X.pkl")
)

joblib.dump(
    scaler_y,
    os.path.join(MODEL_DIR, "scaler_y.pkl")
)

print("Model and scalers saved successfully.")

Model and scalers saved successfully.


In [11]:
import os
import numpy as np
import joblib

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

BASE = r"C:\Users\padhi\Downloads\siH26052_data"

FEATURE_DIR = os.path.join(BASE, "features")
MODEL_DIR = os.path.join(BASE, "model")

X_test = np.load(
    os.path.join(FEATURE_DIR, "X_test.npy"),
    mmap_mode="r"
)

y_test = np.load(
    os.path.join(FEATURE_DIR, "y_test.npy"),
    mmap_mode="r"
)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_test: (247998, 257)
y_test: (247998, 257)


In [12]:
model = joblib.load(
    os.path.join(MODEL_DIR, "mlp_model.pkl")
)

scaler_X = joblib.load(
    os.path.join(MODEL_DIR, "scaler_X.pkl")
)

scaler_y = joblib.load(
    os.path.join(MODEL_DIR, "scaler_y.pkl")
)

print("Model loaded successfully.")

Model loaded successfully.


In [13]:
X_test_scaled = scaler_X.transform(X_test)

print("Test data scaled.")
print(X_test_scaled.shape)

Test data scaled.
(247998, 257)


In [14]:
y_test_pred_scaled = model.predict(
    X_test_scaled
)

print("Prediction complete.")
print(y_test_pred_scaled.shape)

Prediction complete.
(247998, 257)


In [15]:
y_test_pred = scaler_y.inverse_transform(
    y_test_pred_scaled
)

print("Converted predictions back to original scale.")

Converted predictions back to original scale.


In [16]:
test_mse = mean_squared_error(
    y_test,
    y_test_pred
)

test_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

test_r2 = r2_score(
    y_test,
    y_test_pred
)


print("TEST RESULTS")


print("MSE :", test_mse)
print("MAE :", test_mae)
print("R²  :", test_r2)

TEST RESULTS
MSE : 0.06369055062532425
MAE : 0.07701955735683441
R²  : 0.8352088332176208


In [74]:
import os
import numpy as np
import soundfile as sf
import librosa
import joblib

# ============================================================
# PATHS
# ============================================================

BASE = r"C:\Users\padhi\Downloads\siH26052_data"

MODEL_DIR = os.path.join(
    BASE,
    "model"
)

NOISY_DIR = os.path.join(
    BASE,
    "noisy_speeches_test"
)

CLEAN_DIR = os.path.join(
    BASE,
    "test_new",
    "clean"
)

# ============================================================
# LOAD MODEL
# ============================================================

model = joblib.load(
    os.path.join(
        MODEL_DIR,
        "mlp_model.pkl"
    )
)

scaler_X = joblib.load(
    os.path.join(
        MODEL_DIR,
        "scaler_X.pkl"
    )
)

scaler_y = joblib.load(
    os.path.join(
        MODEL_DIR,
        "scaler_y.pkl"
    )
)

print("Model loaded successfully.")

print("Noisy folder :", NOISY_DIR)
print("Clean folder :", CLEAN_DIR)

Model loaded successfully.
Noisy folder : C:\Users\padhi\Downloads\siH26052_data\noisy_speeches_test
Clean folder : C:\Users\padhi\Downloads\siH26052_data\test_new\clean


In [75]:
# ============================================================
# SELECT YOUR INPUT AUDIO FILE
# ============================================================

input_path = input(
    "Enter the path of your noisy WAV file: "
).strip().strip('"')

if not os.path.exists(input_path):

    print("ERROR: File not found!")
    print(input_path)

else:

    noisy_path = input_path

    filename = os.path.basename(
        noisy_path
    )

    # Clean file has the same filename
    clean_path = os.path.join(
        BASE,
        "test_new",
        "clean",
        filename
    )

    print("Testing file:", filename)
    print("Noisy path  :", noisy_path)
    print("Clean path  :", clean_path)

    if not os.path.exists(clean_path):

        print("\nWARNING: Corresponding clean file not found!")

    else:

        print("Clean file found.")

Enter the path of your noisy WAV file:  "C:\Users\padhi\Downloads\siH26052_data\noise_speeches_test\908-31957-0003.wav"


Testing file: 908-31957-0003.wav
Noisy path  : C:\Users\padhi\Downloads\siH26052_data\noise_speeches_test\908-31957-0003.wav
Clean path  : C:\Users\padhi\Downloads\siH26052_data\test_new\clean\908-31957-0003.wav
Clean file found.


In [76]:
# ============================================================
# LOAD AUDIO
# ============================================================

noisy, sr = sf.read(
    noisy_path
)

clean, sr_clean = sf.read(
    clean_path
)

# Stereo → mono
if noisy.ndim > 1:
    noisy = np.mean(
        noisy,
        axis=1
    )

if clean.ndim > 1:
    clean = np.mean(
        clean,
        axis=1
    )

noisy = noisy.astype(
    np.float32
)

clean = clean.astype(
    np.float32
)

print("Sample rate :", sr)
print("Clean rate  :", sr_clean)
print("Noisy samples:", len(noisy))
print("Clean samples:", len(clean))

Sample rate : 16000
Clean rate  : 16000
Noisy samples: 105040
Clean samples: 105040


In [77]:
# ============================================================
# STFT
# ============================================================

N_FFT = 512
HOP_LENGTH = 256

noisy_stft = librosa.stft(
    noisy,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH
)

noisy_mag = np.abs(
    noisy_stft
)

noisy_phase = np.angle(
    noisy_stft
)

print(
    "STFT shape:",
    noisy_stft.shape
)

X = noisy_mag.T

print(
    "Input shape:",
    X.shape
)

STFT shape: (257, 411)
Input shape: (411, 257)


In [78]:
# ============================================================
# SCALE INPUT
# ============================================================

X_scaled = scaler_X.transform(
    X
)

print(
    "Scaled input shape:",
    X_scaled.shape
)

# ============================================================
# MODEL PREDICTION
# ============================================================

pred_scaled = model.predict(
    X_scaled
)

print(
    "Prediction shape:",
    pred_scaled.shape
)

Scaled input shape: (411, 257)
Prediction shape: (411, 257)


In [79]:
# ============================================================
# INVERSE SCALE
# ============================================================

pred_mag = scaler_y.inverse_transform(
    pred_scaled
)

print(
    "Predicted magnitude shape:",
    pred_mag.shape
)

# Frames × frequency
# → frequency × frames

pred_mag = pred_mag.T

print(
    "Predicted magnitude:",
    pred_mag.shape
)

print(
    "Noisy phase:",
    noisy_phase.shape
)

# Prevent negative magnitude
pred_mag = np.maximum(
    pred_mag,
    0
)

Predicted magnitude shape: (411, 257)
Predicted magnitude: (257, 411)
Noisy phase: (257, 411)


In [80]:
# ============================================================
# RECONSTRUCT ENHANCED STFT
# ============================================================

enhanced_stft = (
    pred_mag *
    np.exp(1j * noisy_phase)
)

# ============================================================
# ISTFT
# ============================================================

enhanced_audio = librosa.istft(
    enhanced_stft,
    hop_length=HOP_LENGTH
)

# Match original length

if len(enhanced_audio) > len(noisy):

    enhanced_audio = enhanced_audio[
        :len(noisy)
    ]

elif len(enhanced_audio) < len(noisy):

    enhanced_audio = np.pad(
        enhanced_audio,
        (
            0,
            len(noisy) - len(enhanced_audio)
        )
    )

enhanced_audio = enhanced_audio.astype(
    np.float32
)

print(
    "Enhanced audio samples:",
    len(enhanced_audio)
)

print("Enhancement complete.")

Enhanced audio samples: 105040
Enhancement complete.


In [81]:
# ============================================================
# SNR FUNCTION
# ============================================================

def calculate_snr(clean, signal):

    min_len = min(
        len(clean),
        len(signal)
    )

    clean_temp = clean[:min_len]
    signal_temp = signal[:min_len]

    noise = clean_temp - signal_temp

    signal_power = np.mean(
        clean_temp ** 2
    )

    noise_power = np.mean(
        noise ** 2
    )

    if noise_power <= 1e-12:
        return float("inf")

    return 10 * np.log10(
        signal_power /
        noise_power
    )


# ============================================================
# ORIGINAL TARGET SNR
# ============================================================

original_snr = 5.0


# ============================================================
# ENHANCED SNR
# ============================================================

enhanced_snr = calculate_snr(
    clean,
    enhanced_audio
)

improvement = (
    enhanced_snr -
    original_snr
)


# ============================================================
# SAVE WITH UNIQUE NUMBER
# ============================================================

counter = 1

while True:

    output_filename = (
        f"enhanced_{counter}.wav"
    )

    output_path = os.path.join(
        OUTPUT_DIR,
        output_filename
    )

    if not os.path.exists(
        output_path
    ):
        break

    counter += 1


sf.write(
    output_path,
    enhanced_audio,
    sr
)


# ============================================================
# DISPLAY
# ============================================================

print("\n")
print("=" * 50)
print("       SPEECH ENHANCEMENT RESULTS")
print("=" * 50)

print(
    "\nInput file     :",
    os.path.basename(noisy_path)
)

print(
    "\nOriginal SNR   : {:.2f} dB".format(
        original_snr
    )
)

print(
    "Enhanced SNR   : {:.2f} dB".format(
        enhanced_snr
    )
)

print(
    "Improvement    : {:+.2f} dB".format(
        improvement
    )
)

print("\nEnhanced file  :")
print(output_path)

print("\n" + "=" * 50)



       SPEECH ENHANCEMENT RESULTS

Input file     : 908-31957-0003.wav

Original SNR   : 5.00 dB
Enhanced SNR   : 8.01 dB
Improvement    : +3.01 dB

Enhanced file  :
C:\Users\padhi\Downloads\siH26052_data\enhanced\enhanced_5.wav

